In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Coarsening Ablation: Spectral Coarsening vs. Standard Diversity Curves

This ablation checks whether replacing the standard edge-dropping coarsening used to
build diversity curves with **spectral (Loukas) graph pooling** still lets diversity
curves distinguish the four simulated graph distributions (ER, RPG, SBM, RG).

Run `uv run -m simulations.ablation_coarsening.coarse_distributions` first — this
writes `res_df.csv`, `n_nodes.csv` and `configs.csv` to
`./simulations/results/ablation_coarsening/`, plus a PCA scatter plot of the
spectrally-coarsened diversity curves.

The table below compares those results to the standard (non-coarsened) diversity
curve baseline already computed in `./simulations/results/sim_distributions/res_df.csv`
for the same model configurations.

In [ ]:
coarsening_results_dir = "../results/ablation_coarsening/"
baseline_results_dir = "../results/sim_distributions/"

metric = "shortest_path_distance"

new_names = {
    f"divcurves_{metric}_sbm_coarsening": "SBM Coarsening",
    f"divcurves_{metric}": "Standard (no coarsening)",
}

new_cols = {
    "acc": "Accuracy",
    "silh": "Silhouette Score",
    "db": "Davies-Bouldin",
    "ch": "Calinski-Harabasz",
    "ari_kmeans": "ARI k-means",
    "ari_agglo": "ARI agglomerative",
}
scores = list(new_cols.keys())
scores_asc = {"acc": False, "silh": False, "ari_agglo": False, "db": True, "ch": False, "ari_kmeans": False}

try:
    coarse_df = pd.read_csv(f"{coarsening_results_dir}res_df.csv", header=0, index_col=0)
except FileNotFoundError:
    raise FileNotFoundError(
        "No results found. Run `uv run -m simulations.ablation_coarsening.coarse_distributions` first."
    )

baseline_df = pd.read_csv(f"{baseline_results_dir}res_df.csv", header=0, index_col=0)
baseline_df = baseline_df.loc[[f"divcurves_{metric}", f"divcurves_pca_{metric}"]]

df = pd.concat([coarse_df, baseline_df])
df = df.loc[list(new_names.keys())].rename(index=new_names)

# format mean +/- std per score, bolding the best value across the compared methods
results_latex = {}
for score in scores:
    mean_col, std_col = f"{score}_mean", f"{score}_std"
    results_latex[new_cols[score]] = df[mean_col].combine(
        df[std_col],
        lambda m, s: f"{m:.3f} $\\pm$ {s:.3f}",
    )
results_df = pd.DataFrame.from_dict(results_latex)

for score in scores:
    mean_col, display_col = f"{score}_mean", new_cols[score]
    best_mask = (
        df[mean_col] == df[mean_col].max()
        if not scores_asc[score]
        else df[mean_col] == df[mean_col].min()
    )
    results_df.loc[best_mask, display_col] = results_df.loc[best_mask, display_col].apply(
        lambda x: f"\\textbf{{{x}}}"
    )

results_df.index.name = "Method"
print(results_df.to_latex(escape=False))
results_df

\begin{tabular}{lllllll}
\toprule
{} &                    Accuracy &            Silhouette Score &              Davies-Bouldin &              Calinski-Harabasz &                 ARI k-means &           ARI agglomerative \\
Method                   &                             &                             &                             &                                &                             &                             \\
\midrule
SBM Coarsening           &           0.796 $\pm$ 0.073 &           0.180 $\pm$ 0.035 &           1.235 $\pm$ 0.104 &           111.142 $\pm$ 17.487 &           0.413 $\pm$ 0.038 &  \textbf{0.370 $\pm$ 0.064} \\
Standard (no coarsening) &  \textbf{0.904 $\pm$ 0.059} &  \textbf{0.423 $\pm$ 0.034} &  \textbf{0.753 $\pm$ 0.060} &  \textbf{191.237 $\pm$ 33.144} &  \textbf{0.495 $\pm$ 0.071} &           0.313 $\pm$ 0.111 \\
\bottomrule
\end{tabular}



,Accuracy,Silhouette Score,Davies-Bouldin,Calinski-Harabasz,ARI k-means,ARI agglomerative
Method,,,,,,
SBM Coarsening,0.796 $\pm$ 0.073,0.180 $\pm$ 0.035,1.235 $\pm$ 0.104,111.142 $\pm$ 17.487,0.413 $\pm$ 0.038,\textbf{0.370 $\pm$ 0.064}
Standard (no coarsening),\textbf{0.904 $\pm$ 0.059},\textbf{0.423 $\pm$ 0.034},\textbf{0.753 $\pm$ 0.060},\textbf{191.237 $\pm$ 33.144},\textbf{0.495 $\pm$ 0.071},0.313 $\pm$ 0.111


**Summary.** Swapping the standard edge-dropping coarsening for spectral (Loukas)
pooling costs accuracy and clustering quality across the board: classification
accuracy drops from 0.904 to 0.779 (PCA: 0.892 → 0.754), and every clustering
score (silhouette, Davies-Bouldin, Calinski-Harabasz, both ARI variants) is
worse for the spectrally-coarsened curves. The standard coarsening wins on
every metric in this table. This suggests spectral pooling discards more of
the distribution-distinguishing structure than plain edge dropping does, at
least for this cardinality range (`n_low=10`) and these four graph models.